# STAR testing - Hamilton STAR / STARLet

Validation notebook for the v1 STAR (`pylabrobot.hamilton.star`).

It drives a `STARDevice` - the instrument as a resource, with its deck as its child. The driver
stays reachable underneath as `star.driver`, and machine-level reads that the device does not
proxy are sent through it.

**`setup()` moves the machine.** Watch the log: it reports each phase at `DEBUG` and the machine
it found at `INFO`. It runs in three steps:

1. **discover** - read-only. Machine configuration, arm geometry, channel count, and every
   channel's firmware, width and installed hardware.
2. **initialize** - `C0 VI` on a machine that is not initialized, which homes every drive; or
   `C0 ZA` alone on one that is, to raise the channels to Z safety.
3. **capability initialization** - the channels eject whatever is mounted on them, including grippers.

If you want to connect and look without anything moving, run `discover()` on its own; the cell
below shows how.

Set `protocol_mode` to `"simulation"` to run every cell against a simulated STAR - no hardware,
no USB, and the same code paths as the real driver.

## 1- Run identity

In [1]:
# --- Run identity ---
protocol_mode = "execution"  # simulation OR execution
user_name = "star_user"
run_identifier = "star_v1_validation"

# --- Which instrument ---
# One of the factories in pylabrobot.hamilton.star.device. It fixes the machine's footprint and
# where its deck sits inside it, and builds the matching deck.
instrument = "STAR"  # STAR OR STARLet OR STAR_with_extension_housing

# --- Device selection (only needed with more than one Hamilton on USB) ---
device_address = None  # USB address, e.g. 3
serial_number = None  # USB serial, e.g. "1234567"

# --- Motion ---
# The X-arm move near the end of this notebook only runs when this is True.
allow_x_arm_move = False

# --- 96-head ---
# Where the 96-head ejects when it is initialized: head channel A1, in deck mm. Initializing it
# throws off whatever is mounted, so this has to be somewhere tips may be dropped, which depends
# on where the waste sits on this deck - hence no default. Setup initializes the head when this is
# set, and reports that it cannot when it is None. This machine was last sent (-263.8, 108.3,
# 200.0), read off the `C0 EI` command in an earlier run.
head96_tip_discard_location = (-263.8, 108.3, 200.0)  # as a Coordinate below, after imports

## 2- Imports

In [2]:
from pylabrobot.hamilton.star.device import STAR, STAR_with_extension_housing, STARLet
from pylabrobot.hamilton.star.driver.features.head96 import Head96
from pylabrobot.hamilton.star.driver.master import STARDriver
from pylabrobot.resources.coordinate import Coordinate

## 3- Logging

Uses PyLabRobot's own `setup_logger`, exactly as every other PLR run does: a single
date-stamped file per day, appended to across runs. Both the file and the notebook are at
`IO` level, so every byte sent to and received from the machine is visible and recorded.

Re-running this cell is safe: `setup_logger` replaces the file handler and `verbose`
replaces the console handler, rather than stacking a second one of each.

In [3]:
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

log_dir = f"_logs/{protocol_mode}"

# PLR's own logger setup: one date-stamped file per day, appended to across runs. Re-running this
# cell replaces the file handler rather than stacking a second one, so lines are never duplicated.
pylabrobot.setup_logger(log_dir, level=LOG_LEVEL_IO)

# Console at IO level too: every byte sent and received appears in the notebook.
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

print(f"appending to {log_dir}/pylabrobot-<YYYYMMDD>.log")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

appending to _logs/execution/pylabrobot-<YYYYMMDD>.log


2026-08-19 15:58:42,971 - pylabrobot - INFO - --- star_v1_validation (execution) ---


## 4- Connect and bring the machine up

In simulation this is a `STARSimulationDriver`, which answers as a real instrument does - the
same command assembly, error decoding and response parsing run either way.

Set `head96_tip_discard_location` above for setup to initialize the 96-head too; without it, setup
brings everything else up and reports that it could not do the head.

To connect **without moving anything**, replace `await star.setup()` with:

```python
await star._open()
star._connected = True
await star.discover()
```

In [4]:
build = {
  "STAR": STAR,
  "STARLet": STARLet,
  "STAR_with_extension_housing": STAR_with_extension_housing,
}[instrument]

# The instrument builds its own deck and hands it to the driver, which models the machine into it.
# A simulated one answers from that model, so it is built here rather than passed in.
if protocol_mode == "execution":
  star = build(driver=STARDriver(device_address=device_address, serial_number=serial_number))
else:
  star = build(simulation=True)

# Setup builds each capability the machine turns out to have, but not over one that is already
# there - so a capability configured here keeps its configuration. In simulation the head is
# already there and answers for itself, so configure that one rather than replacing it.
if head96_tip_discard_location is not None:
  if star.driver.head96 is None:
    star.driver.head96 = Head96(star.driver)
  star.head96.configuration.tip_discard_location = Coordinate(*head96_tip_discard_location)

await star.setup()

print(star)
# setup logs this summary at INFO; printed here too so it is the first thing you see.
print(star.driver.format_setup_summary())

2026-08-19 15:58:42,993 - pylabrobot.hamilton.star.driver.master - DEBUG - Setting up STAR on USB 0x08af:0x8000 ...
2026-08-19 15:58:42,995 - pylabrobot.io.usb - INFO - Finding USB device...
2026-08-19 15:58:43,016 - pylabrobot.io.usb - INFO - Found USB device.
2026-08-19 15:58:43,019 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-08-19 15:58:46,022 - pylabrobot.hamilton.star.drive

Hamilton STAR(STARDriver, 56-track deck)
[Hamilton STAR] Connected on USB 0x08af:0x8000
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Configuration: 54 slots
  Autoload: 1D barcode scanner
  Arms: 1
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 1340.2 mm, workspace -323.2 to 1517.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


In [5]:
deck = star.deck
deck

HamiltonSTARDeck(name='deck', location=Coordinate(121.800, 116.000, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

In [6]:
star.driver.deck

HamiltonSTARDeck(name='deck', location=Coordinate(121.800, 116.000, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

## 5- Has this firmware stack been driven before?

What the machine is made of is captured in full by the capture script; what a run wants to know here
is whether any of its boards report firmware this driver has not been driven against.


In [7]:
from pylabrobot.hamilton.star.driver.confirmed_firmware_versions import suggest_entry, unconfirmed

# Has each of this machine's boards been driven on the firmware it reports?
new = unconfirmed(star.driver.firmware)
print()
if not new:
  print(f"firmware: all {len(star.driver.firmware)} capabilities confirmed")
else:
  print(f"firmware: {len(new)} of {len(star.driver.firmware)} capabilities not seen before.")
  print("if this machine works, add them to confirmed_firmware_versions.py:")
  for capability, version in new.items():
    print(suggest_entry(capability, version))

2026-08-19 15:58:49,233 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QWid0065'


wash stations         : 1=False  2=False
tip waste x           : 1340.0 mm
iSWAP collision-free  : 350.0 to 1140.0 mm
pip maximal y         : 606.5 mm


2026-08-19 15:58:49,287 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QWid0065er00/00qw1')


initialized           : True

firmware: all 6 capabilities confirmed


## 6- Sensor read: tip presence

Each channel's sleeve sensor reports whether a tip is mounted. This reads sensors; it does not
move a channel. After a full setup every channel should be empty: the channel initialization
ejects whatever was on them.

In [10]:
presence = await star.driver.request_tip_presence()
for channel, has_tip in enumerate(presence):
  print(f"  channel {channel}: {'tip' if has_tip else '-'}")

2026-08-19 15:58:49,333 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RTid0066'
2026-08-19 15:58:49,375 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RTid0066er00/00rt0 0 0 0 0 0 0 0')


  channel 0: -
  channel 1: -
  channel 2: -
  channel 3: -
  channel 4: -
  channel 5: -
  channel 6: -
  channel 7: -


## 7- The front cover

Two read-only commands, and what they mean is exactly what this check is for.

`C0 RW` reports three inputs, the first of them the cover input. `C0 QC` reports the cover
position. Neither says whether a cover is *fitted*: the master acts only on its non-volatile
configuration, so `main_front_cover_monitoring_installed` is what decides whether the cover is
watched at all, and `star.front_cover` exists only when it is set.

This machine reports it as not installed while the cover and its switch are physically there, so
`QC` is sent raw below rather than through the capability.

**Run this three times and record what changes**: cover shut, cover open, and cover cable
disconnected. If the cover input tracks the position it is a position input; if it holds while
the position changes it is a presence input; if neither moves, the master is not reading the
switch at all - which is what a configuration that says the monitoring is not installed predicts.

That last outcome is the one that decides whether `FrontCover` is worth keeping: a machine that
answers nothing here has no cover to drive, and the capability would only ever be an empty
`request_position` on machines configured differently from this one.


In [11]:
cover_input, second_input, reserve_input = await star.driver.request_cover_input_status()
print(f"inputs        : cover={cover_input}  second={second_input}  reserve={reserve_input}")

c = star.driver.configuration
print(
  f"monitoring    : main={c.main_front_cover_monitoring_installed}"
  f"  additional={c.additional_front_cover_monitoring_installed}"
)
print(f"covers        : left={c.left_cover_installed}  right={c.right_cover_installed}")
print(f"capability    : {star.front_cover}")

# C0 QC - request cover position. Read-only, and sent raw so it answers even on a machine whose
# configuration says the monitoring is not installed.
print(f"position (raw): {await star.driver.send_raw_command('C0QCid9989')}")
if star.front_cover is not None:
  print(f"position      : {await star.front_cover.request_position()}")

2026-08-19 15:58:49,385 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RWid0067'
2026-08-19 15:58:49,408 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RWid0067er00/00rw000')
2026-08-19 15:58:49,412 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QCid9989'


inputs        : cover=False  second=False  reserve=False
monitoring    : main=False  additional=False
covers        : left=False  right=False
capability    : None


2026-08-19 15:58:49,432 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QCid9989er00/00qc1')


position (raw): C0QCid9989er00/00qc1


## 8- Move the X-arm

**This moves the arm and everything mounted on it.** Only run it with the deck clear along the
path, and only after setup has raised the channels to Z safety.

Gated on `allow_x_arm_move`, set at the top of the notebook.

In [12]:
arm = star.x_arm
print(f"travel range: {arm.configuration.x_range} mm")

target = 500.0
if allow_x_arm_move:
  await arm.move_x(target)
  print(f"moved to {target} mm")
else:
  print(f"skipped. set allow_x_arm_move = True to move to {target} mm")

# out-of-range targets are refused before anything reaches the wire
try:
  await arm.move_x(5000.0)
except ValueError as e:
  print("guard:", e)

travel range: (95.0, 1340.2) mm
skipped. set allow_x_arm_move = True to move to 500.0 mm
guard: left X-arm x=5000.0mm is outside its drive travel range [95.0, 1340.2].


In [16]:
# The same gate as the section above: this moves the arm.
allow_x_arm_move = True
if allow_x_arm_move:
  await star.x_arm.move_x(500.0)
  print("moved to 500.0 mm")
else:
  print("skipped. set allow_x_arm_move = True to move")

# What the machine says, and where the model puts the arm's reference point. They should agree.
position = await star.x_arm.request_position()
arm_resource = deck.get_resource("left_x_arm")
seated = arm_resource.get_location_wrt(deck)
print(f"machine: {position} mm")
print(f"model  : {seated.x + arm_resource.get_anchor(x=star.x_arm.reference_anchor).x} mm")

2026-08-19 15:58:49,553 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0XPid0071la05000lr3lw7'
2026-08-19 15:58:51,068 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0XPid0071er00')
2026-08-19 15:58:51,073 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0072'
2026-08-19 15:58:51,083 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0072rx+05002 +000050017')
2026-08-19 15:58:51,086 - pylabrobot.hamilton.star.driver.features.x_arm - DEBUG - the left X-arm was sent to 500.0 mm and came to rest at 500.2 mm
2026-08-19 15:58:51,090 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'X0RXid0073'
2026-08-19 15:58:51,098 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'X0RXid0073rx+05001 +000050013')


moved to 500.0 mm
machine: 500.1 mm
model  : 500.1 mm


## 9- What the rig settled, and what the driver does about it

These were open questions until 2026-08-19; the sections that answered them are gone, and what they
found is in the driver. Kept here so nobody re-derives them.

**An X move always arrives.** Every condition, every repeat, ends exactly on target. What looked
like a 0.2 to 0.4 mm error was a position read taken before the arm had stopped: the move's reply
comes when the move ends. `move_x` now reads until two reads in a row find the arm at the target.

**How the arm gets there depends on the acceleration.** At index 3 it approaches and never
overshoots; at index 5 it swings past and comes back, in five runs of five. A 96-head parked
forward makes the swing bigger - 0.42 against 0.32 mm - which is the arm carrying an off-centre
mass. It all settles in 27 to 90 ms.

**The master and the X-drive board agree** on where the arm is, so reading `X0 RX` is equivalent to
reading `C0 RX`, and neither `X0 XP` nor `C0 JX` closes a position loop the other does not.

**The autoload's scanner resolves 0.1 mm per increment**, measured as exactly 22.5 mm per track
between tracks 10 and 20, and confirmed by the unit's own
configuration, which discovery now reads rather than assuming. Its drive counts from track 1, a
hundred millimetres along the deck: at track 10 it reads 202.5 mm, exactly nine tracks, so its zero
sits on track 1 itself rather than half a track off it.

**The front cover is watched by an input, not by `C0 QC`.** With the monitoring bit clear, `C0 RW`
char 1 followed the cover - 1 shut, 0 open - while `QC` answered "closed" both times. Whether `QC`
works on a machine that declares the monitoring is still open, and needs the configuration write in
section 13.


## 10- Where does the 96-head's permitted Y area actually start?

`H0 YA` takes 6000 to 36000 increments - 93.75 to 562.5 mm - in every firmware document from 2013
on. This machine refuses the low end with error 58, "Y drive position outside of permitted area",
so what the parameter accepts and what the machine allows are two different things. The head shares
the arm with the pipetting channels, which is the obvious thing that would constrain it.

This finds the front limit by bisection: a position the machine takes, one it refuses, and halving
until they meet. Each probe is a real move of a few millimetres.

**This moves the 96-head**, forward along Y and then home in Y and Z when it parks. The deck has
to be clear along its sweep. It refuses to start unless the head is retracted to its safety height,
since Y travel happens at whatever Z the head is at. Gated on `allow_x_arm_move`.


In [25]:
if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to run")
elif star.head96 is None:
  print("no 96-head on this machine")
else:
  head = star.head96
  # The head travels in Y at whatever Z it is at, so it must be retracted first: raised, it sweeps
  # over what is on the deck; low, it sweeps through it.
  z_now = await head.request_z_position()
  z_range = head.configuration.z_range
  if z_range is None:
    raise RuntimeError("the head's Z window was not probed; run setup before this")
  if z_now < z_range[1] - 1.0:
    raise RuntimeError(
      f"the head sits at z={z_now} mm, below its safety height of {z_range[1]} mm - retract it "
      "with head.move_to_safe_z() before sweeping it in Y"
    )
  print(f"head is retracted at z={z_now} mm")

  y_low, y_high = head.configuration.y_range
  print(f"documented travel: {y_low} to {y_high} mm")
  print(f"the drive reports it is at {await head.request_y_position():.2f} mm")

  accepted, refused = (y_low + y_high) / 2, y_low  # the middle works; the documented low does not
  await head.move_y(accepted)
  print(f"  {accepted:7.2f} mm accepted, drive reports {await head.request_y_position():7.2f} mm")

  for _ in range(8):
    if accepted - refused <= 0.5:
      break
    probe = (accepted + refused) / 2
    try:
      await head.move_y(probe)
    except Exception as error:  # noqa: BLE001 - the machine says what it will not do
      refused = probe
      print(
        f"  {probe:7.2f} mm refused, drive reports {await head.request_y_position():7.2f} mm"
        f"  ({error})"
      )
      continue
    accepted = probe
    print(f"  {probe:7.2f} mm accepted, drive reports {await head.request_y_position():7.2f} mm")

  print(f"\nthe permitted area starts between {refused:.2f} and {accepted:.2f} mm")
  print(f"the documented minimum is {y_low}, so {accepted - y_low:.2f} mm of it is out of reach")

  await head.park()
  print(f"\nparked; the drive reports {await head.request_y_position():.2f} mm")

2026-08-19 16:00:48,389 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RZid0847'
2026-08-19 16:00:48,399 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RZid0847rz+67401 +67393')
2026-08-19 16:00:48,402 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0848'
2026-08-19 16:00:48,412 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0848ry+28500 +28496')
2026-08-19 16:00:48,414 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0849rayv'


head is retracted at z=336.97 mm
documented travel: 93.75 to 562.5 mm
the drive reports it is at 445.25 mm


2026-08-19 16:00:48,438 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0849yv25000')
2026-08-19 16:00:48,442 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0850rayr'
2026-08-19 16:00:48,451 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0850yr35000')
2026-08-19 16:00:48,454 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0YAid0851ya21000yv25000yr35000yw15'
2026-08-19 16:00:49,465 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0851er00')
2026-08-19 16:00:49,470 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0852'
2026-08-19 16:00:49,479 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0852ry+21000 +21005')
2026-08-19 16:00:49,483 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0853'
2026-08-19 16:00:49,492 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0853ry+21000 +21005')
2026-08-19 16:00:49,496 - pylabrobot.io.usb - IO - [0x

   328.12 mm accepted, drive reports  328.20 mm


2026-08-19 16:00:50,530 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0856er00')
2026-08-19 16:00:50,535 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0857'
2026-08-19 16:00:50,545 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0857ry+13500 +13506')
2026-08-19 16:00:50,549 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0858'
2026-08-19 16:00:50,559 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0858ry+13500 +13506')
2026-08-19 16:00:50,563 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0859rayv'
2026-08-19 16:00:50,572 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0859yv25000')
2026-08-19 16:00:50,576 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0860rayr'
2026-08-19 16:00:50,586 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0860yr35000')
2026-08-19 16:00:50,589 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write

...

   104.74 mm accepted, drive reports  104.83 mm


2026-08-19 16:00:53,095 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0YAid0886er00')
2026-08-19 16:00:53,100 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0887'
2026-08-19 16:00:53,109 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0887ry+06586 +06590')
2026-08-19 16:00:53,112 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0888'
2026-08-19 16:00:53,122 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0888ry+06586 +06590')
2026-08-19 16:00:53,126 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0889rayv'
2026-08-19 16:00:53,135 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0889yv25000')
2026-08-19 16:00:53,138 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RAid0890rayr'
2026-08-19 16:00:53,148 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RAid0890yr35000')
2026-08-19 16:00:53,152 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write

   102.91 mm accepted, drive reports  102.97 mm
   101.99 mm refused, drive reports  102.97 mm  ({'CoRe 96 Head': UnknownHamiltonError('Y drive position outside of permitted area')}, H0YAid0891er58)

the permitted area starts between 101.99 and 102.91 mm
the documented minimum is 93.75, so 9.16 mm of it is out of reach


2026-08-19 16:00:55,245 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0MOid0894er00')
2026-08-19 16:00:55,250 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0895'
2026-08-19 16:00:55,260 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0895ry+35485 +35482')
2026-08-19 16:00:55,263 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'H0RYid0896'
2026-08-19 16:00:55,273 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'H0RYid0896ry+35485 +35482')



parked; the drive reports 554.41 mm


## 11- The instrument configuration, and what writing it would mean

`C0 AK` writes the machine's non-volatile configuration. It takes **21 parameters**, each with a
default, and the master's convention is that an unsent parameter takes its default - so a partial
`AK` does not change one field, it rewrites all of them. Sending `kb` alone would declare no
channels, no 96-head, a different arm width and a different waste position.

Every one of the 21 is readable: `C0 RM` answers `kb` and `kp`, `C0 QM` the other 19. The cell below
reads them, rebuilds the command that would restore exactly what the machine says today, and shows
what changes if the front cover monitoring bit is set. **It sends nothing.**

Why we would want to: with `kb` bit 2 clear, `C0 QC` answered `qc1` with the cover open, so the
master is not reading the switch. Setting the bit is the only way to find out whether `QC` reports
the cover on a machine that declares the monitoring - and it is also what makes the machine abort a
run when the cover opens, which is why it was turned off in the first place.


In [27]:
import re

# The 21 parameters AK takes, in the order the specification lists them.
AK_PARAMETERS = "ka ke xt xa xw kb xl xn xr xo xm xx xu xv kp ys kl km ym yu yx".split()


def read_fields(reply: str) -> dict:
  """The two-letter fields in a reply, as the machine wrote them."""
  return dict(re.findall(r"([a-z]{2})([0-9A-Fa-f]+)", reply.split("er00/00", 1)[-1]))


if protocol_mode != "execution":
  raise SystemExit("nothing to read: a simulated machine has no configuration to rebuild")

machine = await star.driver.send_command(module="C0", command="RM")
extended = await star.driver.send_command(module="C0", command="QM")
read = {**read_fields(extended), **read_fields(machine)}

missing = [name for name in AK_PARAMETERS if name not in read]
print(f"read {len(AK_PARAMETERS) - len(missing)} of {len(AK_PARAMETERS)} parameters")
if missing:
  print(f"MISSING, so a safe write is not possible: {missing}")
else:
  as_it_stands = "".join(f"{name}{read[name]}" for name in AK_PARAMETERS)
  print(f"\nrestores exactly what the machine says now:\n  C0AK{as_it_stands}")

  with_monitoring = dict(read)
  with_monitoring["kb"] = f"{int(read['kb'], 16) | 0b100:02X}"
  proposed = "".join(f"{name}{with_monitoring[name]}" for name in AK_PARAMETERS)
  print(f"\nwith the front cover monitoring bit set:\n  C0AK{proposed}")
  print(f"\nkb {read['kb']} -> {with_monitoring['kb']}")
  print(
    "everything else identical:",
    as_it_stands.replace(f"kb{read['kb']}", "")
    == proposed.replace(f"kb{with_monitoring['kb']}", ""),
  )

2026-08-19 16:02:49,719 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0RMid2277'
2026-08-19 16:02:49,780 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0RMid2277er00/00kb0Bkp08 C00000 X00000 P10000 P20000 P30000 P40000 P50000 P60000 P70000 P80000 I00000 R00000 H00000')
2026-08-19 16:02:49,784 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] write: b'C0QMid2278'
2026-08-19 16:02:49,822 - pylabrobot.io.usb - IO - [0x8af:0x8000][][] read: bytearray(b'C0QMid2278er00/00ka010003xt54xa54xw13400xl07xr00xm03500xx11400ys090xu3540xv3700yu0060kl360kc0yx0060ke00000000xn00xo00ym6065kr0km360')


read 21 of 21 parameters

restores exactly what the machine says now:
  C0AKka010003ke00000000xt54xa54xw13400kb0Bxl07xn00xr00xo00xm03500xx11400xu3540xv3700kp08ys090kl360km360ym6065yu0060yx0060

with the front cover monitoring bit set:
  C0AKka010003ke00000000xt54xa54xw13400kb0Fxl07xn00xr00xo00xm03500xx11400xu3540xv3700kp08ys090kl360km360ym6065yu0060yx0060

kb 0B -> 0F
everything else identical: True


### Writing it

Only with `allow_configuration_write = True`, set in the cell itself so it cannot be reached by
running the notebook top to bottom. It writes the full command built above, reads the configuration
back, and prints the restore command in case the read-back does not match.

Keep the restore line from the cell above. If anything goes wrong, sending it puts the machine back.


In [28]:
allow_configuration_write = False

if not allow_configuration_write:
  print("skipped. this rewrites the machine's non-volatile configuration")
elif missing:
  print("refused: not every parameter could be read")
else:
  print(f"restore command, keep this:\n  C0AK{as_it_stands}\n")
  print(await star.driver.send_raw_command(f"C0AK{proposed}"))

  after = {
    **read_fields(await star.driver.send_command(module="C0", command="QM")),
    **read_fields(await star.driver.send_command(module="C0", command="RM")),
  }
  for name in AK_PARAMETERS:
    if after.get(name) != with_monitoring.get(name):
      print(f"  {name}: wrote {with_monitoring.get(name)}, reads back {after.get(name)}")
  print(
    "read back identical to what was written:",
    all(after.get(n) == with_monitoring.get(n) for n in AK_PARAMETERS),
  )

skipped. this rewrites the machine's non-volatile configuration


## 12- What the autoload holds in its own memory

Read-only, nothing moves. Four reads the driver did not have until now, each replacing something it
was assuming.

`request_module_configuration` is the one that matters. Its first field is the scanner's step size -
0.1 or 0.125 mm depending on the unit - which the driver used to hardcode; its second says whether
the loading indicators are fitted. Discovery reads both now, so this section checks that the read
works on hardware and that it agrees with what this machine was assumed to be. Last run answered
`au0 0 0 0 0`, so: 0.1 mm per step, indicators fitted.

The other three are diagnostic. `request_adjustment_status` says whether this autoload has ever been
adjusted - an unadjusted module holds factory defaults rather than its own values, and nothing
derived from them means much. `request_init_slot` gives the track the X drive homes against.
`request_adjustment_values` returns the whole adjustment block unparsed, because the 96-head's
equivalent came back truncated and the shape is worth seeing before anyone writes a parser.

`request_parameter` reads any of the 56 named parameters the module stores. A few worth having are
below; the full sweep belongs in the capture script, not here.


In [ ]:
autoload = star.driver.autoload
if autoload is None:
  print("no autoload on this machine")
else:
  c = autoload.configuration

  # What discovery already read off this unit, and what the driver would have assumed without it.
  step, indicators = await autoload.request_module_configuration()
  print(f"scanner step      : {step} mm  (discovery stored {c.x_drive_mm_per_increment})")
  print(f"loading indicators: {'fitted' if indicators else 'none'}")
  if step != 0.1:
    print("  ! this unit is NOT the 0.1 mm generation - every autoload distance depended on that")

  adjusted_on, adjusted = await autoload.request_adjustment_status()
  print(f"adjusted          : {adjusted} on {adjusted_on}")
  if not adjusted:
    print("  ! unadjusted: its stored values are factory defaults, not this unit's")

  print(f"X drive homes at  : track {await autoload.request_init_slot()}")
  print(f"adjustment block  : {await autoload.request_adjustment_values()}")

  # Named parameters worth having: each drive's stored initialization position, and the barcode
  # reading geometry the carrier loads use.
  for name in ("kx", "ky", "kz", "bi", "bp", "bw", "cn", "co", "yl"):
    try:
      print(f"  {name} -> {await autoload.request_parameter(name)}")
    except Exception as e:  # noqa: BLE001 - a name this firmware does not know is an answer too
      print(f"  {name} -> refused: {e}")

## 13- Does `C0 QA` really answer 0 between tracks?

`request_track` is documented as returning the track the carrier handler sits at, "or 0 when it is
at neither end of a track". Nothing has ever checked that, and two things depend on it: a caller
using 0 to mean "between tracks" rather than "track 0", and the simulator, which cannot produce a
0 at all because it answers from a track it was last told to go to.

The sled is moved to a track, then to a position between two tracks, and `QA` is asked at each. If
the sentence holds, the second read answers 0. The last probes walk away from a track in 2 mm steps
to find how wide the band is that still reports a track - which is the number a caller needs if it
ever converts a track to a position.

**This moves the autoload sled** along the front of the deck, which is what `move_to_track` does
during any carrier load. The wheel is raised first, by the same guard those loads use. Gated on
`allow_autoload_move`, set in the cell itself so running the notebook top to bottom does not move
anything.


In [ ]:
allow_autoload_move = False

if not allow_autoload_move:
  print("skipped. set allow_autoload_move = True to move the sled")
elif star.driver.autoload is None:
  print("no autoload on this machine")
else:
  autoload = star.driver.autoload
  track = 10
  on_track = deck.rails_to_location(track).x
  pitch = deck.rails_to_location(track + 1).x - on_track
  print(f"track {track} sits at {on_track} mm, tracks are {pitch} mm apart\n")

  await autoload.move_to_track(track)
  print(
    f"  on track {track:2d}      : QA says {await autoload.request_track():2d}, "
    f"drive at {await autoload.request_x_position():7.2f} mm"
  )

  # Halfway to the next one, which is as far from either as it is possible to be.
  await autoload.move_x(on_track + pitch / 2)
  between = await autoload.request_track()
  print(
    f"  halfway to {track + 1:2d}    : QA says {between:2d}, "
    f"drive at {await autoload.request_x_position():7.2f} mm"
  )
  print(f"\n  the documented 0 between tracks: {'holds' if between == 0 else 'DOES NOT HOLD'}\n")

  # How far off a track it can be and still be reported as on it.
  for offset in (2.0, 4.0, 6.0, 8.0, 10.0):
    await autoload.move_x(on_track + offset)
    reported = await autoload.request_track()
    print(f"  {offset:4.1f} mm past {track:2d}: QA says {reported:2d}")
    if reported != track:
      print(f"  -> still reported as track {track} up to somewhere under {offset} mm past it")
      break

  await autoload.park()
  print(
    f"\n  parked; QA says {await autoload.request_track()}, "
    f"drive at {await autoload.request_x_position():.2f} mm"
  )

## 14- Raw command escape hatch

Anything not yet wrapped in a named method can be sent directly. **Only send commands you have
confirmed are read-only** - this bypasses every guard in the driver.

In [ ]:
# C0 RF - request the master's firmware version. Read-only.
print(await star.driver.send_command(module="C0", command="RF"))

# the same thing as a raw string, id included
# print(await star.driver.send_raw_command("C0RFid9999"))

## 15- Teardown


In [ ]:
await star.stop()
print("disconnected. connected:", star.driver.connected)

# The log is append-only and stays open for the rest of the session - nothing to close.